# CAPAR 2.0 — Episode Analysis Evaluation

Notebook ini berfungsi untuk menganalisis dan memvisualisasikan data Episode dari MongoDB yang diekspor menggunakan script `exportEpisodeColab.js`.

Notebook ini mencakup:
1. **Trajectory Score (S(t)) vs Thresholds ($\tau_{in}, \tau_{out}$)**
2. **Multimodal Analysis (HR & HRV SDNN/RMSSD)**
3. **Z-Scores Dynamics**
4. **Signal Quality Audit**
5. **Clinical Evaluations (Ablation E1-E6)**

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

print("✅ Libraries loaded.")

✅ Libraries loaded.


### 1. Load Ekspor JSON dari `exportEpisodeColab.js`
Pastikan Anda telah mengunggah file `colab_export.json` ke _Files_ Google Colab (kolom sebelah kiri).

In [2]:
try:
    with open('colab_export.json', 'r') as f:
        data = json.load(f)
    print(f"Berhasil memuat Episode: {data.get('episode_id', 'Unknown')}")
    print(f"Onset Time: {data.get('onset_time')}")
    print(f"Total Segments: {len(data.get('trajectory', []))}")
except FileNotFoundError:
    print("❌ colab_export.json tidak ditemukan. Silakan upload terlebih dahulu!")
    # Dummy data untuk demo bila file tidak ada
    data = {
        "tau_in": 1.86, "tau_out": 1.20,
        "analysis_results": {
             "latent_severity": 4.5,
             "quality_score": 0.95,
             "evaluations": {"E1_Statistical": {"result": "PASS", "score": 2.3}},
             "z_scores_at_peak": {"z_hr": 3.5, "z_rr": 2.1}
        },
        "trajectory": []
    }

❌ colab_export.json tidak ditemukan. Silakan upload terlebih dahulu!


### 2. Clinical Evaluations & Episode Meta

In [3]:
ar = data.get('analysis_results', {})
print("==============================================")
print(" EPISODE ANALYSIS METADATA")
print("==============================================")
print(f"Latent Severity : {ar.get('latent_severity', 'N/A')}")
print(f"Quality Score   : {ar.get('quality_score', 0)*100:.1f}%")
print("\n--- Z-SCORES AT PEAK ---")
z_peak = ar.get('z_scores_at_peak', {})
print(f"Z-Score HR : {z_peak.get('z_hr', 'N/A')}")
print(f"Z-Score RR : {z_peak.get('z_rr', 'N/A')}")

print("\n--- ABLATION EVALUATIONS (E1-E6) ---")
evals = ar.get('evaluations', {})
for k, v in evals.items():
    res = v.get('result', 'N/A')
    sc = v.get('score', 0)
    print(f"- {k}: {res} (Score: {sc:.2f})")
print("==============================================")

 EPISODE ANALYSIS METADATA
Latent Severity : 4.5
Quality Score   : 95.0%

--- Z-SCORES AT PEAK ---
Z-Score HR : 3.5
Z-Score RR : 2.1

--- ABLATION EVALUATIONS (E1-E6) ---
- E1_Statistical: PASS (Score: 2.30)


### 3. Ekstraksi Trajectory ke Pandas DataFrame

In [4]:
if data['trajectory']:
    df = pd.DataFrame(data['trajectory'])
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values('timestamp')
    
    # Extract nested fields safely
    df['hr'] = df.get('hr', np.nan)
    
    # Extract HRV 
    if 'hrv' in df.columns:
        df['sdnn'] = df['hrv'].apply(lambda x: x.get('sdnn') if isinstance(x, dict) else np.nan)
        df['rmssd'] = df['hrv'].apply(lambda x: x.get('rmssd') if isinstance(x, dict) else np.nan)
        df['dfa'] = df['hrv'].apply(lambda x: x.get('dfa') if isinstance(x, dict) else np.nan)
        
    # Extract Z-Scores
    if 'z_scores' in df.columns:
        df['z_hr'] = df['z_scores'].apply(lambda x: x.get('z_hr') if isinstance(x, dict) else np.nan)
        df['z_rr'] = df['z_scores'].apply(lambda x: x.get('z_rr') if isinstance(x, dict) else np.nan)
        
    display(df.head())
else:
    print("Trajectory data kosong.")

Trajectory data kosong.


### 4. Visualisasi Trajectory Multimodal

In [5]:
if data['trajectory']:
    fig, axs = plt.subplots(4, 1, figsize=(14, 12), sharex=True, gridspec_kw={'height_ratios': [2, 1, 1, 0.5]})
    
    # Formatter waktu
    xfmt = mdates.DateFormatter('%H:%M:%S')
    
    # 1. Main Anomaly Score S(t)
    ax0 = axs[0]
    ax0.plot(df['timestamp'], df['score'], color='#087F7A', linewidth=2.5, marker='o', markersize=4, label='S(t) Anomaly Score')
    ax0.axhline(y=data['tau_in'], color='#C62828', linestyle='--', linewidth=1.5, label=f"Tau In ({data['tau_in']})")
    ax0.axhline(y=data['tau_out'], color='#EF8D00', linestyle='-.', linewidth=1.5, label=f"Tau Out ({data['tau_out']})")
    
    ax0.fill_between(df['timestamp'], df['score'], color='#087F7A', alpha=0.1)
    ax0.set_ylabel('Score', fontweight='bold')
    ax0.set_title('Episode Trajectory Analysis', fontweight='bold', fontsize=14)
    ax0.grid(True, linestyle=':', alpha=0.7)
    ax0.legend(loc='upper right')
    
    # 2. Multimodal HR & Z-Scores
    ax1 = axs[1]
    ax1_2 = ax1.twinx()
    if 'z_hr' in df.columns:
        ax1.plot(df['timestamp'], df['z_hr'], color='#1976D2', linewidth=2, label='Z-HR')
    if 'hr' in df.columns:
        ax1_2.plot(df['timestamp'], df['hr'], color='#E53935', linewidth=1.5, linestyle=':', label='Raw HR (bpm)')
        ax1_2.set_ylabel('HR (bpm)', color='#E53935')
        
    ax1.set_ylabel('Z-Score HR', fontweight='bold', color='#1976D2')
    ax1.grid(True, linestyle=':', alpha=0.7)
    
    # 3. HRV Features (SDNN)
    ax2 = axs[2]
    if 'sdnn' in df.columns:
        ax2.plot(df['timestamp'], df['sdnn'], color='#8E24AA', linewidth=2, label='SDNN (ms)')
    ax2.set_ylabel('HRV (ms)', fontweight='bold', color='#8E24AA')
    ax2.grid(True, linestyle=':', alpha=0.7)
    
    # 4. Signal Quality Track
    ax3 = axs[3]
    if 'signal_quality' in df.columns:
        # Map quality ke color/value
        color_map = {'Valid': '#81C784', 'Artifact': '#E57373', 'Missing': '#E0E0E0'}
        df['sq_val'] = df['signal_quality'].map({'Valid': 1, 'Artifact': 0, 'Missing': 0}).fillna(1)
        df['sq_col'] = df['signal_quality'].map(color_map).fillna('#81C784')
        
        ax3.bar(df['timestamp'], height=1, width=0.001, color=df['sq_col'], align='center')
        ax3.set_yticks([])
        ax3.set_ylabel('Signal Quality', fontweight='bold')
        
    # Format X-axis
    axs[3].xaxis.set_major_formatter(xfmt)
    plt.setp(axs[3].get_xticklabels(), rotation=30, ha="right")
    
    plt.tight_layout()
    plt.show()
